# Examen de Big Data - Spark ML: Clasificacion Multiclase de Desercion Estudiantil

**Autor:** Andrés Felipe Maestre Badillo
**ID:** U00071243

---

## Seccion A. Carga, limpieza, ingenieria de caracteristicas y particion (Tareas 1 a 3)

### 1. Carga y exploracion de datos

En esta primera celda de codigo realizamos la configuracion inicial y la exploracion del conjunto de datos:

- **Descarga automatica:** Descargamos desde Kaggle el dataset Higher Education Predictors of Student Retention usando kagglehub y lo guardamos como data.csv.
- **Creacion de la SparkSession:** Configuramos 4 GB de memoria en el driver (spark.driver.memory = 4g) y desactivamos la propagacion de restricciones (spark.sql.constraintPropagation.enabled = false) para evitar que el optimizador de Spark colapse en Google Colab durante el entrenamiento.
- **Carga y exploracion:** Leemos el archivo CSV activando la deteccion automatica de tipos (inferSchema=True) y encabezados (header=True). Mostramos el esquema de variables, el total de filas y columnas, y la tabla con la distribucion de frecuencias y porcentajes de la variable objetivo Target.

In [1]:
import glob
import os
import shutil
import kagglehub
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Descargar el dataset con tu código de kagglehub
path = kagglehub.dataset_download(
    "thedevastator/higher-education-predictors-of-student-retention"
)
print("Path to dataset files:", path)

# Copiar el CSV descargado a "data.csv" en la carpeta actual de Colab
archivo_origen = glob.glob(os.path.join(path, "*.csv"))[0]
shutil.copy(archivo_origen, "data.csv")
print("Archivo copiado exitosamente como: data.csv\n")

# 2. Crear la SparkSession con la configuración exigida en el examen
spark = (
    SparkSession.builder.appName("dropout_sparkml")
    .config("spark.driver.memory", "4g")
    .config(
        "spark.sql.constraintPropagation.enabled", "false"
    )  # evita explosión del optimizador
    .getOrCreate()
)

# 3. Cargar data.csv con Spark (este dataset de thedevastator usa coma ',' como separador)
df_raw = spark.read.csv("data.csv", header=True, inferSchema=True, sep=",")

# Mostrar esquema
print("=== ESQUEMA DEL DATASET ===")
df_raw.printSchema()

# Número de filas y columnas
num_filas = df_raw.count()
num_cols = len(df_raw.columns)
print(f"Número total de filas: {num_filas}")
print(f"Número total de columnas: {num_cols}\n")

# Distribución de clases (conteo y porcentaje por Target)
print("=== DISTRIBUCIÓN DE CLASES (Target) ===")
dist_clases = (
    df_raw.groupBy("Target")
    .count()
    .withColumn("porcentaje", F.round((F.col("count") / num_filas) * 100, 2))
    .orderBy(F.desc("count"))
)
dist_clases.show()

100%|██████████| 87.2k/87.2k [00:00<00:00, 468kB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/thedevastator/higher-education-predictors-of-student-retention/versions/2
Archivo copiado exitosamente como: data.csv

=== ESQUEMA DEL DATASET ===
root
 |-- Marital status: integer (nullable = true)
 |-- Application mode: integer (nullable = true)
 |-- Application order: integer (nullable = true)
 |-- Course: integer (nullable = true)
 |-- Daytime/evening attendance: integer (nullable = true)
 |-- Previous qualification: integer (nullable = true)
 |-- Nacionality: integer (nullable = true)
 |-- Mother's qualification: integer (nullable = true)
 |-- Father's qualification: integer (nullable = true)
 |-- Mother's occupation: integer (nullable = true)
 |-- Father's occupation: integer (nullable = true)
 |-- Displaced: integer (nullable = true)
 |-- Educational special needs: integer (nullable = true)
 |-- Debtor: integer (nullable = true)
 |-- Tuition fees up to date: integer (nullable = true)
 |-- Gender: integer (nullable = true)
 |

### Analisis del desbalance de clases (Cierre del Punto 1)

Al analizar la distribucion de la variable objetivo Target en la salida anterior, se observa un desbalance importante entre las tres categorias:

- **Clase dominante (Graduate):** Es la clase mayoritaria con 2.209 estudiantes (49.93 % del total), representando casi la mitad de los registros.
- **Clase intermedia (Dropout):** Cuenta con 1.421 estudiantes (32.12 % del total).
- **Clase minoritaria (Enrolled):** Es la clase con menor representacion, con solo 794 estudiantes (17.95 % del total).

**Por que importa este desbalance:**
Si entrenamos un modelo buscando optimizar unicamente la exactitud global (Accuracy), el algoritmo aprendera a favorecer la clase mayoritaria (Graduate) y cometera muchos errores en la clase minoritaria (Enrolled) y en los estudiantes en riesgo de desercion (Dropout), que son justamente los casos criticos que la universidad necesita detectar a tiempo.

---

### 2. Preprocesamiento, estrategia de desbalance y particion

En la siguiente celda de codigo ejecutamos el preprocesamiento de los datos:

- **Normalizacion de nombres de columnas:** Utilizamos una expresion regular (re.sub) para convertir todos los nombres de las columnas a minusculas y reemplazar espacios, parentesis y simbolos por guiones bajos.
- **Limpieza y verificacion:** Revisamos la cantidad de valores nulos, eliminamos filas duplicadas o nulas con dropDuplicates() y dropna(), y verificamos que los tipos de datos sean correctos.
- **Indexado de la variable objetivo fuera del Pipeline:** Aplicamos StringIndexer sobre la columna target para generar la columna numerica label (0, 1, 2) y guardamos el orden exacto de las etiquetas (Graduate, Dropout, Enrolled) en labels.json para traducir las predicciones en la aplicacion de Streamlit.
- **Justificacion de la estrategia para el desbalance:** Implementamos dos medidas complementarias:
  1. Calculamos una columna de pesos por clase (class_weight) inversamente proporcional a la frecuencia de cada categoria, para que los modelos penalicen con mayor fuerza los errores en las clases minoritarias.
  2. Elegimos la metrica F1 Ponderado (f1) para la validacion cruzada, ya que equilibra Precision y Recall teniendo en cuenta el peso de cada clase.
- **Particion de datos:** Dividimos el dataset en conjunto de entrenamiento (80 %) y conjunto de prueba (20 %) usando randomSplit([0.8, 0.2], seed=42).

In [2]:
import json
import re
from pyspark.ml.feature import StringIndexer

# 1. Normalizar nombres de columnas con expresión regular (minúsculas, sin espacios ni símbolos)
df = df_raw
for col_name in df.columns:
  nuevo_nombre = re.sub(r"[^a-z0-9]+", "_", col_name.strip().lower()).strip("_")
  df = df.withColumnRenamed(col_name, nuevo_nombre)

print("=== 1. NOMBRES DE COLUMNAS NORMALIZADOS ===")
print(df.columns)

# 2. Revisar y eliminar duplicados y nulos, y verificar tipos de datos
conteo_nulos = df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
).collect()[0]
total_nulos = sum(conteo_nulos.asDict().values())

filas_antes = df.count()
df = df.dropDuplicates().dropna()
filas_despues = df.count()

print(f"\n=== 2. LIMPIEZA DE DUPLICADOS Y NULOS ===")
print(f"Total de valores nulos encontrados: {total_nulos}")
print(
    f"Filas antes: {filas_antes} | Filas después de eliminar duplicados y"
    f" nulos: {filas_despues}"
)

print("\n=== TIPOS DE DATOS VERIFICADOS ===")
for nombre, tipo in df.dtypes:
  print(f" - {nombre}: {tipo}")

# 3. Indexar la etiqueta Target FUERA del pipeline y guardar el orden de labels
label_indexer = StringIndexer(
    inputCol="target", outputCol="label", handleInvalid="keep"
)
label_indexer_model = label_indexer.fit(df)
labels_orden = label_indexer_model.labels

print(f"\n=== 3. INDEXADO DE TARGET (labels) ===")
print(
    "Orden guardado de clases (índice -> clase):"
    f" {list(enumerate(labels_orden))}"
)

# Guardamos el orden en un archivo JSON para usarlo luego en Streamlit
with open("labels.json", "w", encoding="utf-8") as f:
  json.dump(labels_orden, f)

df = label_indexer_model.transform(df)

# 4. Estrategia para el desbalance: calcular pesos por clase (weightCol = "class_weight")
conteo_por_label = df.groupBy("label").count().collect()
total_filas = float(filas_despues)
num_clases = float(len(labels_orden))

mapa_pesos = {
    row["label"]: round(total_filas / (num_clases * row["count"]), 4)
    for row in conteo_por_label
}
print(
    "\n=== 4. PESOS POR CLASE PARA EL DESBALANCE (label -> peso) ==="
    f"\n{mapa_pesos}"
)

expr_pesos = F.create_map([F.lit(x) for kv in mapa_pesos.items() for x in kv])
df = df.withColumn("class_weight", expr_pesos[F.col("label")])

# 5. Dividir en entrenamiento (80%) y prueba (20%) con semilla fija 42
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"\n=== 5. PARTICIÓN ENTRENAMIENTO / PRUEBA (80/20, seed=42) ===")
print(f"Filas en entrenamiento (train_df): {train_df.count()}")
print(f"Filas en prueba (test_df): {test_df.count()}")

=== 1. NOMBRES DE COLUMNAS NORMALIZADOS ===
['marital_status', 'application_mode', 'application_order', 'course', 'daytime_evening_attendance', 'previous_qualification', 'nacionality', 'mother_s_qualification', 'father_s_qualification', 'mother_s_occupation', 'father_s_occupation', 'displaced', 'educational_special_needs', 'debtor', 'tuition_fees_up_to_date', 'gender', 'scholarship_holder', 'age_at_enrollment', 'international', 'curricular_units_1st_sem_credited', 'curricular_units_1st_sem_enrolled', 'curricular_units_1st_sem_evaluations', 'curricular_units_1st_sem_approved', 'curricular_units_1st_sem_grade', 'curricular_units_1st_sem_without_evaluations', 'curricular_units_2nd_sem_credited', 'curricular_units_2nd_sem_enrolled', 'curricular_units_2nd_sem_evaluations', 'curricular_units_2nd_sem_approved', 'curricular_units_2nd_sem_grade', 'curricular_units_2nd_sem_without_evaluations', 'unemployment_rate', 'inflation_rate', 'gdp', 'target']

=== 2. LIMPIEZA DE DUPLICADOS Y NULOS ===
Tot

### 3. Ingenieria de caracteristicas y seleccion de variables

En la siguiente celda de codigo construimos nuevas variables con sentido academico, optimizamos el numero de particiones y justificamos la seleccion de caracteristicas:

- **Creacion de 3 variables derivadas (controlando la division por cero con F.when):**
  1. **tasa_aprobacion_sem1:** Divide las materias aprobadas entre las materias inscritas en el primer semestre (si las inscritas son 0, asigna 0.0). Mide el nivel de exito academico inicial del estudiante al ingresar a la universidad.
  2. **tasa_aprobacion_sem2:** Divide las materias aprobadas entre las materias inscritas en el segundo semestre (controlando division por cero). Mide la consolidacion o deterioro del rendimiento academico al cerrar el primer ano.
  3. **variacion_nota_semestres:** Resta el promedio de nota del primer semestre al promedio del segundo semestre (nota_sem2 - nota_sem1). Mide la evolucion o tendencia del promedio del estudiante entre ambos periodos (valor positivo indica mejora; valor negativo indica caida en el rendimiento).
- **Control de particiones (Maximo 4):** Aplicamos coalesce(4) sobre train_df y test_df tal como indica la guia del examen, evitando sobrecarga de tareas pequenas durante el entrenamiento en Google Colab.
- **Justificacion de la seleccion de variables (featureImportances de Random Forest):** Entrenamos un Random Forest exploratorio sobre el conjunto de entrenamiento para calcular la importancia de todas las variables y seleccionamos las caracteristicas categoricas y numericas con mayor poder predictivo (incluyendo las 3 variables derivadas). Esto reduce el ruido, acelera la validacion cruzada y permite construir un formulario claro en Streamlit.
- **Separacion de columnas categoricas y numericas:** Definimos las listas cat_cols y num_cols que entraran al Pipeline en el Punto 4 (donde se aplicaran StringIndexer con handleInvalid="keep", OneHotEncoder, VectorAssembler y StandardScaler).

In [3]:
import pandas as pd
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler


# 1. Función para crear las 3 variables derivadas controlando la división por cero
def agregar_variables_derivadas(dataframe):
  return (
      dataframe.withColumn(
          "tasa_aprobacion_sem1",
          F.when(
              F.col("curricular_units_1st_sem_enrolled") > 0,
              F.col("curricular_units_1st_sem_approved")
              / F.col("curricular_units_1st_sem_enrolled"),
          ).otherwise(0.0),
      )
      .withColumn(
          "tasa_aprobacion_sem2",
          F.when(
              F.col("curricular_units_2nd_sem_enrolled") > 0,
              F.col("curricular_units_2nd_sem_approved")
              / F.col("curricular_units_2nd_sem_enrolled"),
          ).otherwise(0.0),
      )
      .withColumn(
          "variacion_nota_semestres",
          F.col("curricular_units_2nd_sem_grade")
          - F.col("curricular_units_1st_sem_grade"),
      )
  )


# Aplicar a train_df y test_df y limitar a máximo 4 particiones (recomendación del examen)
train_df = agregar_variables_derivadas(train_df).coalesce(4).cache()
test_df = agregar_variables_derivadas(test_df).coalesce(4).cache()

print("=== 1. MUESTRA DE LAS 3 VARIABLES DERIVADAS CREADAS ===")
train_df.select(
    "curricular_units_1st_sem_enrolled",
    "curricular_units_1st_sem_approved",
    F.round("tasa_aprobacion_sem1", 3).alias("tasa_aprob_sem1"),
    F.round("tasa_aprobacion_sem2", 3).alias("tasa_aprob_sem2"),
    F.round("variacion_nota_semestres", 2).alias("variacion_nota"),
    "target",
).show(5)

print(
    "Particiones en train_df:"
    f" {train_df.rdd.getNumPartitions()} | Particiones en test_df:"
    f" {test_df.rdd.getNumPartitions()}"
)

# 2. Justificar la selección de variables con featureImportances de Random Forest
todas_candidatas = [
    c for c in train_df.columns if c not in ("target", "label", "class_weight")
]
assembler_temp = VectorAssembler(
    inputCols=todas_candidatas, outputCol="features_temp", handleInvalid="keep"
)
train_temp = assembler_temp.transform(train_df)

rf_selector = RandomForestClassifier(
    featuresCol="features_temp",
    labelCol="label",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=6,
    seed=42,
)
modelo_selector = rf_selector.fit(train_temp)

tabla_importancia = (
    pd.DataFrame({
        "Variable": todas_candidatas,
        "Importancia": modelo_selector.featureImportances.toArray(),
    })
    .sort_values(by="Importancia", ascending=False)
    .reset_index(drop=True)
)

print(
    "\n=== 2. TOP 15 VARIABLES SEGÚN IMPORTANCIA (Random Forest"
    " featureImportances) ==="
)
print(tabla_importancia.head(15).to_string(index=False))

# 3. Separar columnas categóricas y numéricas seleccionadas para el Pipeline
cat_cols = [
    "tuition_fees_up_to_date",  # Matrícula al día (0=No, 1=Sí)
    "scholarship_holder",  # Becado (0=No, 1=Sí)
    "debtor",  # Deudor (0=No, 1=Sí)
    "gender",  # Género (0=Femenino, 1=Masculino)
    "displaced",  # Desplazado (0=No, 1=Sí)
]

num_cols = [
    "age_at_enrollment",
    "curricular_units_1st_sem_enrolled",
    "curricular_units_1st_sem_approved",
    "curricular_units_1st_sem_grade",
    "curricular_units_2nd_sem_enrolled",
    "curricular_units_2nd_sem_approved",
    "curricular_units_2nd_sem_grade",
    "tasa_aprobacion_sem1",
    "tasa_aprobacion_sem2",
    "variacion_nota_semestres",
]

print(f"\n=== 3. VARIABLES SELECCIONADAS PARA EL PIPELINE ===")
print(f"Columnas Categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Columnas Numéricas ({len(num_cols)}): {num_cols}")

=== 1. MUESTRA DE LAS 3 VARIABLES DERIVADAS CREADAS ===
+---------------------------------+---------------------------------+---------------+---------------+--------------+--------+
|curricular_units_1st_sem_enrolled|curricular_units_1st_sem_approved|tasa_aprob_sem1|tasa_aprob_sem2|variacion_nota|  target|
+---------------------------------+---------------------------------+---------------+---------------+--------------+--------+
|                                5|                                4|            0.8|            1.0|         -0.75|Graduate|
|                                6|                                6|            1.0|            1.0|          1.17|Graduate|
|                                7|                                6|          0.857|          0.857|           0.0|Graduate|
|                                7|                                0|            0.0|            0.0|           0.0| Dropout|
|                                6|                           

---

## Seccion B. Pipelines, hiperparametrizacion y validacion cruzada (Tareas 4 y 5)

### 4 y 5. Construccion de Pipelines completos y ajuste de hiperparametros

En la siguiente celda de codigo construimos y entrenamos 2 modelos de clasificacion multiclase de Spark ML (siguiendo la recomendacion del examen de entrenar 2 modelos por tiempo de ejecucion):

- **Etapas dentro de cada Pipeline:**
  1. **StringIndexer (con handleInvalid="keep"):** Convierte las columnas categoricas en indices numericos manteniendo categorias no vistas.
  2. **OneHotEncoder:** Codifica los indices categoricos en vectores binarios (One-Hot).
  3. **VectorAssembler:** Une las variables categoricas codificadas y las variables numericas (incluyendo las 3 variables derivadas) en un solo vector llamado raw_features.
  4. **StandardScaler:** Escala las caracteristicas (con desviacion estandar) en la columna final features.
  5. **Clasificador Multiclase:**
     - **Modelo 1 - LogisticRegression(family="multinomial"):** Ajustamos 2 hiperparametros en la grilla: regParam (0.01, 0.1) y elasticNetParam (0.0, 0.5).
     - **Modelo 2 - RandomForestClassifier(seed=42):** Ajustamos 2 hiperparametros en la grilla: numTrees (50, 100) y maxDepth (5, 10).
- **Validacion Cruzada (CrossValidator):** Para cada modelo usamos ParamGridBuilder().build() y CrossValidator con numFolds=3, semilla fija (seed=42) y el evaluador MulticlassClassificationEvaluator con la metrica F1 ponderado (metricName="f1").

In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder


# 1. Función para crear las etapas de preprocesamiento dentro del Pipeline
def crear_etapas_pipeline(cat_columns, num_columns):
  idx_cols = [f"{c}_idx" for c in cat_columns]
  ohe_cols = [f"{c}_ohe" for c in cat_columns]

  indexer = StringIndexer(
      inputCols=cat_columns, outputCols=idx_cols, handleInvalid="keep"
  )
  encoder = OneHotEncoder(
      inputCols=idx_cols, outputCols=ohe_cols, handleInvalid="keep"
  )
  assembler = VectorAssembler(
      inputCols=ohe_cols + num_columns,
      outputCol="raw_features",
      handleInvalid="keep",
  )
  scaler = StandardScaler(
      inputCol="raw_features",
      outputCol="features",
      withStd=True,
      withMean=False,
  )

  return [indexer, encoder, assembler, scaler]


# Evaluador con F1 ponderado para CrossValidator
evaluador_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

# ==============================================================================
# MODELO 1: LogisticRegression (family="multinomial")
# ==============================================================================
etapas_lr = crear_etapas_pipeline(cat_cols, num_cols)
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",
    family="multinomial",
    maxIter=100,
)
pipeline_lr = Pipeline(stages=etapas_lr + [lr])

param_grid_lr = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

cv_lr = CrossValidator(
    estimator=pipeline_lr,
    estimatorParamMaps=param_grid_lr,
    evaluator=evaluador_f1,
    numFolds=3,
    seed=42,
    parallelism=2,
)

print("Entrenando Modelo 1: LogisticRegression (family='multinomial')...")
cv_model_lr = cv_lr.fit(train_df)
mejor_lr = cv_model_lr.bestModel.stages[-1]

print("=== MEJORES HIPERPARÁMETROS: LOGISTIC REGRESSION ===")
print(f" - F1 promedio en Cross-Validation: {max(cv_model_lr.avgMetrics):.4f}")
print(f" - regParam: {mejor_lr.getRegParam()}")
print(f" - elasticNetParam: {mejor_lr.getElasticNetParam()}\n")

# ==============================================================================
# MODELO 2: RandomForestClassifier
# ==============================================================================
etapas_rf = crear_etapas_pipeline(cat_cols, num_cols)
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",
    seed=42,
)
pipeline_rf = Pipeline(stages=etapas_rf + [rf])

param_grid_rf = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

cv_rf = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluador_f1,
    numFolds=3,
    seed=42,
    parallelism=2,
)

print("Entrenando Modelo 2: RandomForestClassifier...")
cv_model_rf = cv_rf.fit(train_df)
mejor_rf = cv_model_rf.bestModel.stages[-1]

print("=== MEJORES HIPERPARÁMETROS: RANDOM FOREST ===")
print(f" - F1 promedio en Cross-Validation: {max(cv_model_rf.avgMetrics):.4f}")
print(f" - numTrees: {mejor_rf.getNumTrees}")
print(f" - maxDepth: {mejor_rf.getMaxDepth()}")

Entrenando Modelo 1: LogisticRegression (family='multinomial')...
=== MEJORES HIPERPARÁMETROS: LOGISTIC REGRESSION ===
 - F1 promedio en Cross-Validation: 0.7522
 - regParam: 0.01
 - elasticNetParam: 0.5

Entrenando Modelo 2: RandomForestClassifier...
=== MEJORES HIPERPARÁMETROS: RANDOM FOREST ===
 - F1 promedio en Cross-Validation: 0.7623
 - numTrees: 100
 - maxDepth: 10


---

## Seccion C. Evaluacion, seleccion y guardado del mejor modelo (Tarea 6)

### 6. Evaluacion sobre el conjunto de prueba y guardado del PipelineModel

En la siguiente celda de codigo evaluamos los 2 modelos exclusivamente sobre el conjunto de prueba (test_df, nunca sobre el de entrenamiento):

- **Metricas calculadas:** Accuracy (exactitud), F1 ponderado (f1), Precision ponderada (weightedPrecision) y Recall ponderado (weightedRecall).
- **Comparacion y Matriz de Confusion:** Presentamos una tabla comparativa con los resultados de ambos modelos, seleccionamos el mejor modelo considerando el F1 ponderado frente al desbalance de clases, y calculamos su Matriz de Confusion sobre test_df.
- **Guardado del modelo final:** Extraemos el mejor PipelineModel con bestModel, lo guardamos con write().overwrite().save("modelo_final") y lo comprimimos en modelo_final.zip para descargarlo y usarlo en Streamlit.

In [5]:
import shutil


# 1. Función para calcular las 4 métricas sobre el conjunto de prueba (test_df)
def evaluar_modelo_test(nombre, cv_entrenado, df_prueba):
  predicciones = cv_entrenado.bestModel.transform(df_prueba)
  resultados = {"Modelo": nombre}

  metricas = {
      "Accuracy": "accuracy",
      "F1_Ponderado": "f1",
      "Precision_Ponderada": "weightedPrecision",
      "Recall_Ponderado": "weightedRecall",
  }

  for nombre_col, metrica_spark in metricas.items():
    ev = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName=metrica_spark
    )
    resultados[nombre_col] = round(ev.evaluate(predicciones), 4)

  return resultados, predicciones


res_lr, preds_lr = evaluar_modelo_test(
    "LogisticRegression (Multinomial)", cv_model_lr, test_df
)
res_rf, preds_rf = evaluar_modelo_test(
    "RandomForestClassifier", cv_model_rf, test_df
)

# 2. Tabla comparativa de los modelos
tabla_comparativa = pd.DataFrame([res_lr, res_rf]).sort_values(
    by="F1_Ponderado", ascending=False
)
print("=== TABLA COMPARATIVA EN EL CONJUNTO DE PRUEBA (test_df) ===")
print(tabla_comparativa.to_string(index=False))

# 3. Selección del mejor modelo según F1 ponderado
if res_rf["F1_Ponderado"] >= res_lr["F1_Ponderado"]:
  nombre_mejor = "RandomForestClassifier"
  mejor_modelo_final = cv_model_rf.bestModel
  preds_mejor = preds_rf
else:
  nombre_mejor = "LogisticRegression (Multinomial)"
  mejor_modelo_final = cv_model_lr.bestModel
  preds_mejor = preds_lr

print(f"\n>>> MEJOR MODELO SELECCIONADO: {nombre_mejor} <<<")

# 4. Matriz de confusión del mejor modelo sobre test_df
print(f"\n=== MATRIZ DE CONFUSIÓN DEL MEJOR MODELO ({nombre_mejor}) ===")
print(
    f"Orden de clases (label / prediction): {list(enumerate(labels_orden))}\n"
)

matriz_confusion = (
    preds_mejor.groupBy("label")
    .pivot("prediction", list(range(len(labels_orden))))
    .count()
    .na.fill(0)
    .orderBy("label")
)
matriz_confusion.show()

# 5. Guardar el PipelineModel completo en "modelo_final" y crear modelo_final.zip
mejor_modelo_final.write().overwrite().save("modelo_final")
shutil.make_archive("modelo_final", "zip", "modelo_final")
print(
    "Guardado exitoso: carpeta 'modelo_final' creada y comprimida en"
    " 'modelo_final.zip'."
)

=== TABLA COMPARATIVA EN EL CONJUNTO DE PRUEBA (test_df) ===
                          Modelo  Accuracy  F1_Ponderado  Precision_Ponderada  Recall_Ponderado
          RandomForestClassifier    0.7426        0.7518               0.7727            0.7426
LogisticRegression (Multinomial)    0.7392        0.7476               0.7666            0.7392

>>> MEJOR MODELO SELECCIONADO: RandomForestClassifier <<<

=== MATRIZ DE CONFUSIÓN DEL MEJOR MODELO (RandomForestClassifier) ===
Orden de clases (label / prediction): [(0, 'Graduate'), (1, 'Dropout'), (2, 'Enrolled')]

+-----+---+---+---+
|label|  0|  1|  2|
+-----+---+---+---+
|  0.0|342| 11| 65|
|  1.0| 38|223| 52|
|  2.0| 46| 15| 90|
+-----+---+---+---+

Guardado exitoso: carpeta 'modelo_final' creada y comprimida en 'modelo_final.zip'.


### Analisis de los mejores hiperparametros (Cierre del Punto 5)

A partir de la validacion cruzada de 3 pliegues (3-Fold CrossValidation) optimizando el F1 ponderado, obtuvimos los siguientes resultados en entrenamiento:

- **Modelo 1 - LogisticRegression (Multinomial):** Logro un F1 promedio en validacion cruzada de **0.7522** con **regParam = 0.01** y **elasticNetParam = 0.5**. El valor bajo de regParam (0.01) evita subajustar el modelo y conserva el peso predictivo de las tasas de aprobacion semestrales, mientras que elasticNetParam = 0.5 combina en partes iguales la regularizacion L1 (Lasso) y L2 (Ridge), controlando la colinealidad entre las variables de primer y segundo semestre.
- **Modelo 2 - RandomForestClassifier:** Logro el mejor desempeno en validacion cruzada con un F1 promedio de **0.7623** usando **numTrees = 100** y **maxDepth = 10**. Contar con 100 arboles reduce la varianza del ensamble, y una profundidad maxima de 10 niveles permite capturar relaciones no lineales entre el estado financiero del estudiante (matricula al dia, beca, deuda) y su evolucion academica sin caer en sobreajuste.

---

### Justificacion de la seleccion del mejor modelo y analisis de la Matriz de Confusion (Cierre del Punto 6)

Al evaluar ambos modelos sobre el conjunto de prueba (test_df, datos nunca vistos durante el entrenamiento), el **RandomForestClassifier** supera en todas las metricas a la Regresion Logistica Multinomial:

- **F1 Ponderado:** 0.7518 en Random Forest frente a 0.7476 en Logistic Regression.
- **Precision Ponderada:** 0.7727 en Random Forest frente a 0.7666 en Logistic Regression.
- **Accuracy y Recall Ponderado:** 0.7426 en Random Forest frente a 0.7392 en Logistic Regression.

**Por que elegimos RandomForestClassifier considerando el desbalance de clases:**
Dado que las clases estan desbalanceadas (Graduate = 49.93 %, Dropout = 32.12 %, Enrolled = 17.95 %), el **F1 Ponderado (0.7518)** y la **Precision Ponderada (0.7727)** son los indicadores mas confiables porque ponderan el rendimiento en cada categoria segun su soporte real. Ademas, al analizar la **Matriz de Confusion** del Random Forest sobre los 882 estudiantes de prueba:

- **Clase 0 (Graduate - 418 casos reales):** Clasifica correctamente a **342 estudiantes** (81.8 % de acierto) y solo confunde a 11 con desercion.
- **Clase 1 (Dropout - 313 casos reales):** Detecta correctamente a **223 estudiantes en desercion** (71.2 % de deteccion temprana), permitiendo a la universidad intervenir oportunamente sobre la gran mayoria de alumnos en riesgo.
- **Clase 2 (Enrolled - 151 casos reales, clase minoritaria):** Gracias al uso de pesos por clase (class_weight), logra identificar correctamente a **90 estudiantes** (59.6 % de acierto en la clase mas dificil y minoritaria), logrando que el F1 Ponderado supere al Accuracy global (0.7518 vs 0.7426).

Por estas razones, se selecciona y exporta el PipelineModel de **RandomForestClassifier** en la carpeta **modelo_final**.